# VENOZK STUDIO — Colab (Optimized + Proteus)

**Open-source AI Upscaler**  
Real-ESRGAN (Spandrel) · **Proteus V3 (TRT / ONNX)** · Images & Videos · Batch · T4 / L4 friendly

### How to use
1. **Runtime → Change runtime type → T4 GPU** (or L4)
2. Run **Cell 1** once (installs + downloads models)
3. For **Proteus V3**: the 1× TRT engine is supported.  
   Cell 1 will automatically copy `prob-v3-fgnet-fp32-576x672-1x.trt` into `/content/models/`
4. Configure options in **Cell 2** form → Run Cell 2

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venozk/UPSCALER-BY-VENOZK/blob/main/venozk_upscaler_colab.ipynb)


In [ ]:
#@title **Cell 1 — Setup** { display-mode: "form" }
#@markdown Run once per session. Installs packages + downloads Real-ESRGAN + prepares Proteus TRT.

import os, sys, subprocess, time, shutil, glob

def sh(cmd, timeout=300):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)

print("=" * 64)
print("  VENOZK STUDIO — Setup (Real-ESRGAN + Proteus)")
print("=" * 64)

gpu = sh("nvidia-smi -L").stdout.strip()
if gpu:
    print(f"  GPU: {gpu.split(chr(10))[0]}")
else:
    print("  ⚠ No GPU detected. Runtime → Change runtime type → T4 GPU")

print("\n[1/4] Installing packages...")
pkgs = ["spandrel", "tqdm", "pillow", "onnx", "onnxruntime-gpu"]
r = sh(f"{sys.executable} -m pip install -q {' '.join(pkgs)}")
if r.returncode:
    r = sh(f"{sys.executable} -m pip install {' '.join(pkgs)}")
    if r.returncode:
        print(r.stderr[-500:])
sh(f"{sys.executable} -m pip install -q nvidia-tensorrt 2>/dev/null || true")
try:
    import cv2
except ImportError:
    sh(f"{sys.executable} -m pip install -q opencv-python-headless")
print("  ✓ Packages ready")

print("\n[2/4] Downloading Real-ESRGAN models...")
os.makedirs("/content/models", exist_ok=True)
BASE = "https://github.com/xinntao/Real-ESRGAN/releases/download/"
MODELS = {
    "realesr-general-x4v3.pth":       BASE + "v0.2.5.0/realesr-general-x4v3.pth",
    "realesr-general-wdn-x4v3.pth":   BASE + "v0.2.5.0/realesr-general-wdn-x4v3.pth",
    "realesr-animevideov3.pth":       BASE + "v0.2.5.0/realesr-animevideov3.pth",
    "RealESRGAN_x2plus.pth":          BASE + "v0.2.1/RealESRGAN_x2plus.pth",
    "RealESRGAN_x4plus.pth":          BASE + "v0.1.0/RealESRGAN_x4plus.pth",
    "RealESRGAN_x4plus_anime_6B.pth": BASE + "v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
}
def ok_file(path, min_mb=1):
    return os.path.exists(path) and os.path.getsize(path) > min_mb * 1_000_000
for fname, url in MODELS.items():
    path = f"/content/models/{fname}"
    if ok_file(path):
        print(f"  ✓ {fname} ({os.path.getsize(path)//1048576} MB) — cached")
        continue
    print(f"  ↓ {fname} ...", end=" ", flush=True)
    for _ in range(2):
        sh(f"wget -q -O {path} '{url}'", timeout=180)
        if ok_file(path): break
    if ok_file(path):
        print(f"OK ({os.path.getsize(path)//1048576} MB)")
    else:
        print("FAILED")
        if os.path.exists(path): os.remove(path)

print("\n[3/4] Preparing Proteus V3...")
PROTEUS_TRT  = "/content/models/prob-v3-fgnet-fp32-576x672-1x.trt"
PROTEUS_ONNX = "/content/models/prob-v3-fgnet-fp32-576x672-2x.onnx"
for src in (glob.glob("/content/*.trt") + glob.glob("/content/*.onnx") +
            glob.glob("/content/models/*.trt") + glob.glob("/content/models/*.onnx") +
            glob.glob("/home/workdir/artifacts/*.trt") + glob.glob("/home/workdir/artifacts/*.onnx") +
            glob.glob("/home/workdir/attachments/*.trt") + glob.glob("/home/workdir/attachments/*.onnx")):
    dst = f"/content/models/{os.path.basename(src)}"
    if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
        shutil.copy2(src, dst)
        print(f"  ✓ Copied {os.path.basename(src)}")
if ok_file(PROTEUS_TRT, 10):
    print(f"  ✓ Proteus TRT ready ({os.path.getsize(PROTEUS_TRT)//1048576} MB) ← preferred")
elif ok_file(PROTEUS_ONNX, 5):
    print(f"  ✓ Proteus ONNX ready ({os.path.getsize(PROTEUS_ONNX)//1048576} MB)")
else:
    print("  ⚠ No Proteus model found. Upload the .trt (or .onnx) then re-run Cell 1.")

print("\n[4/4] Checking ffmpeg + NVENC...")
r = sh("ffmpeg -hide_banner -encoders 2>/dev/null | grep -c nvenc")
HAS_NVENC = int(r.stdout.strip() or 0) > 0
print(f"  NVENC: {'available' if HAS_NVENC else 'not available'}")
print("\n" + "=" * 64)
print("  [OK] Cell 1 complete. Now run Cell 2.")
print("=" * 64)


In [ ]:
#@title **Cell 2 — Venozk Studio** { display-mode: "form" }

#@markdown ### 📁 Input / Output
MOUNT_DRIVE = False #@param {type:"boolean"}
INPUT_PATH = "" #@param {type:"string"}
#@markdown Leave empty to upload files from your device. Supports images + videos (batch OK).
OUTPUT_DIR = "/content/output" #@param {type:"string"}
AUTO_DOWNLOAD = True #@param {type:"boolean"}
PREVIEW_ONLY = False #@param {type:"boolean"}
#@markdown Preview = process only first frame / image and show before→after.

#@markdown ---
#@markdown ### 🧠 Upscale / Enhance
MODEL = "general-x4v3" #@param ["general-x4v3", "x2plus", "x4plus", "anime-video-v3", "anime-6B", "proteus-v3"]
#@markdown `general-x4v3` = best all-round + denoise · `x2plus/x4plus` = clean photos · `anime-*` = cartoons · **`proteus-v3`** = Topaz-style (TRT 1× preferred)
TARGET_SCALE = 2.0 #@param {type:"slider", min:1, max:4, step:0.5}
DENOISE = 0.5 #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown Denoise only affects `general-x4v3`.
AUTO_X2 = True #@param {type:"boolean"}
TILE = 512 #@param [256, 384, 512, 768, 1024]
FP16 = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🎛️ Post-process
SHARPEN = 0.3 #@param {type:"slider", min:0, max:1.5, step:0.05}
DEHALO = 0.0 #@param {type:"slider", min:0, max:1, step:0.05}
RECOVER_DETAILS = 0.0 #@param {type:"slider", min:0, max:1, step:0.05}

#@markdown ---
#@markdown ### 🖼️ Image output
IMG_FORMAT = "png" #@param ["png", "jpg", "webp"]
IMG_QUALITY = 95 #@param {type:"slider", min:50, max:100, step:1}

#@markdown ---
#@markdown ### 🎬 Video output
PHONE_MP4 = True #@param {type:"boolean"}
VCODEC = "h264 (CPU)" #@param ["h264 (CPU)", "h265 (CPU, smaller)", "h264 NVENC (GPU, fast)", "h265 NVENC (GPU, fast)", "ProRes 422 HQ (editing)"]
CRF = 16 #@param {type:"slider", min:0, max:30, step:1}
PRESET = "slow" #@param ["ultrafast", "fast", "medium", "slow", "veryslow"]
OUTPUT_FPS = 0 #@param {type:"number"}
KEEP_AUDIO = True #@param {type:"boolean"}
START_SEC = 0 #@param {type:"number"}
DURATION_SEC = 0 #@param {type:"number"}

TILE = int(TILE); CRF = int(CRF); IMG_QUALITY = int(IMG_QUALITY)
TARGET_SCALE = float(TARGET_SCALE); DENOISE = float(DENOISE)
SHARPEN = float(SHARPEN); DEHALO = float(DEHALO); RECOVER_DETAILS = float(RECOVER_DETAILS)
OUTPUT_FPS = float(OUTPUT_FPS); START_SEC = float(START_SEC); DURATION_SEC = float(DURATION_SEC)

import os, glob, time, subprocess, cv2, numpy as np, torch
from tqdm.auto import tqdm
from google.colab import files
from google.colab.patches import cv2_imshow

try: HAS_NVENC
except NameError: HAS_NVENC = False

dev = "cuda" if torch.cuda.is_available() else "cpu"
use_half = FP16 and dev == "cuda"
dt = torch.float16 if use_half else torch.float32
print(f"Device: {dev}  |  dtype: {dt}")

IS_PROTEUS = MODEL == "proteus-v3"
even = lambda v: max(2, int(round(v / 2)) * 2)

# ══════════════════════════════════════════════════════════════════
#  PROTEUS V3 — TRT preferred, ONNX fallback, optimized tiling
# ══════════════════════════════════════════════════════════════════
if IS_PROTEUS:
    import onnxruntime as ort

    PROTEUS_TRT  = "/content/models/prob-v3-fgnet-fp32-576x672-1x.trt"
    PROTEUS_ONNX = "/content/models/prob-v3-fgnet-fp32-576x672-2x.onnx"
    for c in [PROTEUS_TRT, "/content/prob-v3-fgnet-fp32-576x672-1x.trt"]:
        if os.path.exists(c) and os.path.getsize(c) > 10_000_000:
            PROTEUS_TRT = c; break

    use_trt  = os.path.exists(PROTEUS_TRT)  and os.path.getsize(PROTEUS_TRT)  > 10_000_000
    use_onnx = os.path.exists(PROTEUS_ONNX) and os.path.getsize(PROTEUS_ONNX) > 5_000_000
    if not use_trt and not use_onnx:
        raise FileNotFoundError(
            "No Proteus model found.\\n"
            "Upload `prob-v3-fgnet-fp32-576x672-1x.trt` (recommended) or the 2x .onnx\\n"
            "into /content/models/ then re-run."
        )

    session = None
    backend = None
    NATIVE_SCALE = 1

    providers = []
    if dev == "cuda":
        providers.append(("TensorrtExecutionProvider", {
            "device_id": 0,
            "trt_max_workspace_size": 4 * 1024**3,
            "trt_fp16_enable": True,
            "trt_engine_cache_enable": True,
            "trt_engine_cache_path": "/content/models/trt_cache",
        }))
        providers.append(("CUDAExecutionProvider", {
            "device_id": 0,
            "arena_extend_strategy": "kNextPowerOfTwo",
            "gpu_mem_limit": 6 * 1024**3,
            "cudnn_conv_algo_search": "EXHAUSTIVE",
            "do_copy_in_default_stream": True,
        }))
    providers.append("CPUExecutionProvider")

    sess_opt = ort.SessionOptions()
    sess_opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_opt.intra_op_num_threads = 4

    if use_trt:
        try:
            session = ort.InferenceSession(PROTEUS_TRT, sess_opt, providers=providers)
            backend = "ORT-TensorRT"
            print(f"  Loaded TRT engine  →  {session.get_providers()}")
        except Exception as e:
            print(f"  TRT load failed ({type(e).__name__}): {e}")
            session = None

    if session is None and use_onnx:
        session = ort.InferenceSession(PROTEUS_ONNX, sess_opt, providers=providers)
        backend = "ORT-CUDA"
        NATIVE_SCALE = 2
        print(f"  Loaded ONNX  →  {session.get_providers()}")

    if session is None:
        raise RuntimeError(
            "Could not load Proteus.\\n"
            "The .trt may have been built for a different GPU (T4 = sm_75).\\n"
            "Upload the matching 2× ONNX or rebuild the engine on this Colab GPU."
        )

    inp = session.get_inputs()[0]
    outp = session.get_outputs()[0]
    IN_NAME, OUT_NAME = inp.name, outp.name
    print(f"  Input : {IN_NAME}  {inp.shape}  {inp.type}")
    print(f"  Output: {OUT_NAME} {outp.shape} {outp.type}")
    print(f"  Backend: {backend}  |  Native scale ×{NATIVE_SCALE}")

    NATIVE_H, NATIVE_W = 576, 672
    OVERLAP = 32

    def _to_model(tile_bgr):
        rgb = cv2.cvtColor(tile_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return np.ascontiguousarray(rgb.transpose(2, 0, 1)[None, ...])

    def _from_model(t):
        arr = np.asarray(t)
        if arr.ndim == 4: arr = arr[0]
        if arr.shape[0] in (1, 3): arr = arr.transpose(1, 2, 0)
        arr = np.clip(arr, 0, 1)
        if arr.shape[2] == 1: arr = np.repeat(arr, 3, axis=2)
        return cv2.cvtColor((arr * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

    def run_tile(bgr_tile):
        if bgr_tile.shape[:2] != (NATIVE_H, NATIVE_W):
            bgr_tile = cv2.resize(bgr_tile, (NATIVE_W, NATIVE_H), interpolation=cv2.INTER_AREA)
        return _from_model(session.run([OUT_NAME], {IN_NAME: _to_model(bgr_tile)})[0])

    def upscale_proteus(bgr):
        h, w = bgr.shape[:2]
        model_h, model_w = h * NATIVE_SCALE, w * NATIVE_SCALE
        target_h, target_w = even(h * TARGET_SCALE), even(w * TARGET_SCALE)

        if h <= NATIVE_H and w <= NATIVE_W:
            o = run_tile(bgr)
            if (o.shape[0], o.shape[1]) != (target_h, target_w):
                interp = cv2.INTER_AREA if target_h < o.shape[0] else cv2.INTER_LANCZOS4
                o = cv2.resize(o, (target_w, target_h), interpolation=interp)
            return o

        step_y = max(1, NATIVE_H - OVERLAP)
        step_x = max(1, NATIVE_W - OVERLAP)
        out = np.zeros((model_h, model_w, 3), np.float32)
        weight = np.zeros((model_h, model_w, 1), np.float32)

        wy = np.ones(NATIVE_H, np.float32)
        wx = np.ones(NATIVE_W, np.float32)
        if OVERLAP > 0:
            ramp = np.linspace(0, 1, OVERLAP, dtype=np.float32)
            wy[:OVERLAP] = ramp; wy[-OVERLAP:] = ramp[::-1]
            wx[:OVERLAP] = ramp; wx[-OVERLAP:] = ramp[::-1]
        w2d = (wy[:, None] * wx[None, :])[..., None]

        for y in range(0, h, step_y):
            for x in range(0, w, step_x):
                y1, x1 = min(y + NATIVE_H, h), min(x + NATIVE_W, w)
                th, tw = y1 - y, x1 - x
                tile = np.zeros((NATIVE_H, NATIVE_W, 3), np.uint8)
                tile[:th, :tw] = bgr[y:y1, x:x1]
                o_tile = run_tile(tile).astype(np.float32)
                oy, ox = y * NATIVE_SCALE, x * NATIVE_SCALE
                oh, ow = th * NATIVE_SCALE, tw * NATIVE_SCALE
                if NATIVE_SCALE == 1:
                    ww = w2d[:th, :tw]
                else:
                    ww = cv2.resize(w2d[:th, :tw, 0], (ow, oh), interpolation=cv2.INTER_LINEAR)[..., None]
                out[oy:oy+oh, ox:ox+ow] += o_tile[:oh, :ow] * ww
                weight[oy:oy+oh, ox:ox+ow] += ww

        out = np.clip(out / np.maximum(weight, 1e-6), 0, 255).astype(np.uint8)
        if (out.shape[0], out.shape[1]) != (target_h, target_w):
            interp = cv2.INTER_AREA if target_h < out.shape[0] else cv2.INTER_LANCZOS4
            out = cv2.resize(out, (target_w, target_h), interpolation=interp)
        return out

    print(f"Proteus ready | native {NATIVE_W}×{NATIVE_H} ×{NATIVE_SCALE} | overlap {OVERLAP}px")
    print(f"  TARGET_SCALE={TARGET_SCALE}× applied after network")

# ══════════════════════════════════════════════════════════════════
#  Real-ESRGAN (Spandrel)
# ══════════════════════════════════════════════════════════════════
else:
    from spandrel import ModelLoader
    FILES = {
        "general-x4v3": "realesr-general-x4v3",
        "x2plus": "RealESRGAN_x2plus",
        "x4plus": "RealESRGAN_x4plus",
        "anime-video-v3": "realesr-animevideov3",
        "anime-6B": "RealESRGAN_x4plus_anime_6B",
    }
    def load_model(name):
        path = f"/content/models/{name}.pth"
        if not os.path.exists(path):
            raise FileNotFoundError(f"Model not found: {path}. Run Cell 1 first.")
        return ModelLoader().load_from_file(path).to(dev).eval()

    model_key = MODEL
    if AUTO_X2 and MODEL == "x4plus" and TARGET_SCALE <= 2:
        model_key = "x2plus"
        print("Auto X2: using native x2plus")

    net = load_model(FILES[model_key])
    if model_key == "general-x4v3" and DENOISE < 1:
        net_wdn = load_model("realesr-general-wdn-x4v3")
        sd_a, sd_b = net.model.state_dict(), net_wdn.model.state_dict()
        mixed = {k: (DENOISE * v + (1 - DENOISE) * sd_b[k] if v.is_floating_point() else v)
                 for k, v in sd_a.items()}
        net.model.load_state_dict(mixed)
        del net_wdn, sd_a, sd_b, mixed
        print(f"Denoise {DENOISE:.2f}: blended")
    if use_half: net = net.half()
    print(f"Loaded {model_key} (native ×{net.scale})")

    @torch.inference_mode()
    def run_net(net, rgb, pad=16):
        s, (h, w) = net.scale, rgb.shape[:2]
        out = np.zeros((h * s, w * s, 3), np.uint8)
        for y in range(0, h, TILE):
            for x in range(0, w, TILE):
                y0, x0 = max(y - pad, 0), max(x - pad, 0)
                y1, x1 = min(y + TILE + pad, h), min(x + TILE + pad, w)
                t = torch.from_numpy(rgb[y0:y1, x0:x1]).to(dev).permute(2, 0, 1)[None].to(dt) / 255
                o = net(t).clamp(0, 1)[0].permute(1, 2, 0).float().mul(255).round().byte().cpu().numpy()
                th, tw = min(TILE, h - y) * s, min(TILE, w - x) * s
                oy, ox = (y - y0) * s, (x - x0) * s
                out[y*s:y*s+th, x*s:x*s+tw] = o[oy:oy+th, ox:ox+tw]
        return out

    def upscale(bgr):
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        o = cv2.cvtColor(run_net(net, rgb), cv2.COLOR_RGB2BGR)
        h, w = bgr.shape[:2]
        W, H = even(w * TARGET_SCALE), even(h * TARGET_SCALE)
        if (o.shape[1], o.shape[0]) != (W, H):
            interp = cv2.INTER_AREA if W < o.shape[1] else cv2.INTER_LANCZOS4
            o = cv2.resize(o, (W, H), interpolation=interp)
        return o

# ── Shared post-process ──────────────────────────────────────────
def apply_dehalo(img, strength):
    if strength <= 0: return img
    soft = cv2.bilateralFilter(img, d=5, sigmaColor=40, sigmaSpace=40)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    edges = np.abs(cv2.Sobel(gray, cv2.CV_32F, 1, 1, ksize=3))
    edges = cv2.GaussianBlur(edges, (0, 0), 1.5)
    edges = edges / (edges.max() + 1e-6)
    mask = (edges * strength)[..., None]
    out = img.astype(np.float32) * (1 - mask) + soft.astype(np.float32) * mask
    return np.clip(out, 0, 255).astype(np.uint8)

def apply_recover_details(img, strength):
    if strength <= 0: return img
    blur = cv2.GaussianBlur(img, (0, 0), 1.2)
    detail = img.astype(np.float32) - blur.astype(np.float32)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrast = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR).astype(np.float32)
    out = img.astype(np.float32) + detail * (strength * 1.4)
    out = out * (1 - strength * 0.35) + contrast * (strength * 0.35)
    return np.clip(out, 0, 255).astype(np.uint8)

def process(bgr):
    o = upscale_proteus(bgr) if IS_PROTEUS else upscale(bgr)
    if DEHALO > 0: o = apply_dehalo(o, DEHALO)
    if RECOVER_DETAILS > 0: o = apply_recover_details(o, RECOVER_DETAILS)
    if SHARPEN > 0:
        o = cv2.addWeighted(o, 1 + SHARPEN, cv2.GaussianBlur(o, (0, 0), 1.0), -SHARPEN, 0)
    return o

def show_cmp(a, b):
    a = cv2.resize(a, (b.shape[1], b.shape[0]), interpolation=cv2.INTER_NEAREST)
    v = np.hstack([a, b])
    k = min(1.0, 1600 / v.shape[1])
    cv2_imshow(cv2.resize(v, None, fx=k, fy=k, interpolation=cv2.INTER_AREA))

def read_image(p):
    im = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    if im is None: raise RuntimeError(f"Cannot read: {p}")
    if im.dtype == np.uint16: im = (im / 257).astype(np.uint8)
    if im.ndim == 2: im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    if im.shape[2] == 4:
        return np.ascontiguousarray(im[:,:,:3]), np.ascontiguousarray(im[:,:,3])
    return im, None

def do_image(p, stem):
    bgr, alpha = read_image(p)
    o = process(bgr)
    if PREVIEW_ONLY: return show_cmp(bgr, o)
    if alpha is not None:
        a = cv2.resize(alpha, (o.shape[1], o.shape[0]), interpolation=cv2.INTER_CUBIC)
        if IMG_FORMAT == "jpg":
            f = (a.astype(np.float32)/255)[...,None]
            o = (o.astype(np.float32)*f + 255*(1-f)).astype(np.uint8)
        else:
            o = np.dstack([o, a])
    q = {"png":[cv2.IMWRITE_PNG_COMPRESSION,3], "jpg":[cv2.IMWRITE_JPEG_QUALITY,IMG_QUALITY],
         "webp":[cv2.IMWRITE_WEBP_QUALITY,IMG_QUALITY]}[IMG_FORMAT]
    out = f"{OUTPUT_DIR}/{stem}_venozk.{IMG_FORMAT}"
    cv2.imwrite(out, o, q)
    return out

def do_video(p, stem):
    global HAS_NVENC
    cap = cv2.VideoCapture(p)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    s0 = int(START_SEC * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, s0)
    n = min(n - s0, int(DURATION_SEC * fps)) if DURATION_SEC > 0 else n - s0
    ok, f = cap.read()
    if not ok: raise RuntimeError("Cannot read video")
    o = process(f)
    if PREVIEW_ONLY:
        cap.release(); return show_cmp(f, o)
    H, W = o.shape[:2]
    V = {
        "h264 (CPU)": ("libx264", ["-crf",str(CRF),"-preset",PRESET,"-pix_fmt","yuv420p"], "mp4"),
        "h265 (CPU, smaller)": ("libx265", ["-crf",str(CRF),"-preset",PRESET,"-pix_fmt","yuv420p","-tag:v","hvc1"], "mp4"),
        "h264 NVENC (GPU, fast)": ("h264_nvenc", ["-cq",str(CRF),"-preset","p5","-pix_fmt","yuv420p"], "mp4"),
        "h265 NVENC (GPU, fast)": ("hevc_nvenc", ["-cq",str(CRF),"-preset","p5","-pix_fmt","yuv420p","-tag:v","hvc1"], "mp4"),
        "ProRes 422 HQ (editing)": ("prores_ks", ["-profile:v","3","-pix_fmt","yuv422p10le"], "mov"),
    }
    vc, va, ext = V[VCODEC]
    if "nvenc" in vc and not HAS_NVENC:
        fallback = "h264 (CPU)" if "264" in vc else "h265 (CPU, smaller)"
        vc, va, ext = V[fallback]
        print(f"  NVENC unavailable → {fallback}")
    if PHONE_MP4 and vc in ("libx264", "h264_nvenc"):
        va = ["-cq",str(CRF),"-preset","p5"] if vc=="h264_nvenc" else ["-crf",str(CRF),"-preset",PRESET]
        va += ["-pix_fmt","yuv420p","-profile:v","high",
               "-level","5.1" if max(W,H)>2048 else "4.2", "-movflags","+faststart"]
        ext = "mp4"
    out = f"{OUTPUT_DIR}/{stem}_venozk.{ext}"
    log_path = f"{OUTPUT_DIR}/.ffmpeg_{stem}.log"
    cmd = ["ffmpeg","-y","-loglevel","error","-f","rawvideo","-pix_fmt","bgr24",
           "-s",f"{W}x{H}","-r",str(fps),"-i","-"]
    if KEEP_AUDIO:
        cmd += ["-ss",str(START_SEC)]
        if DURATION_SEC > 0: cmd += ["-t",str(DURATION_SEC)]
        cmd += ["-i", p]
    cmd += ["-map","0:v"]
    if KEEP_AUDIO: cmd += ["-map","1:a?","-c:a","aac","-b:a","192k","-ac","2"]
    cmd += ["-c:v",vc,*va]
    if OUTPUT_FPS > 0: cmd += ["-r",str(OUTPUT_FPS)]
    cmd += ["-shortest", out]
    logf = open(log_path, "w")
    pr = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=logf)
    broken = False
    try:
        pr.stdin.write(np.ascontiguousarray(o).tobytes())
        for _ in tqdm(range(n-1), desc=stem):
            ok, f = cap.read()
            if not ok: break
            pr.stdin.write(np.ascontiguousarray(process(f)).tobytes())
    except (BrokenPipeError, OSError):
        broken = True
    finally:
        try: pr.stdin.close()
        except: pass
        pr.wait(); cap.release(); logf.close()
    if broken or pr.returncode != 0:
        raise RuntimeError(f"ffmpeg failed (code {pr.returncode})\\n{open(log_path).read()[-800:]}")
    os.remove(log_path)
    return out

# ── collect inputs ───────────────────────────────────────────────
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

IMG_X = {".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"}
VID_X = {".mp4",".mov",".mkv",".avi",".webm",".m4v"}

if not INPUT_PATH.strip():
    uploaded = files.upload()
    paths = [os.path.abspath(n) for n in uploaded]
elif os.path.isdir(INPUT_PATH):
    paths = sorted(glob.glob(INPUT_PATH.rstrip("/") + "/*"))
else:
    paths = [INPUT_PATH]

os.makedirs(OUTPUT_DIR, exist_ok=True)
results = []
t0 = time.time()
for p in paths:
    stem, ext = os.path.splitext(os.path.basename(p))
    ext = ext.lower()
    try:
        if ext in IMG_X: r = do_image(p, stem)
        elif ext in VID_X: r = do_video(p, stem)
        else:
            print("Skipped:", p); continue
        if r:
            results.append(r)
            print("Saved:", r)
    except Exception as e:
        print("FAILED", p, "→", e)
    if dev == "cuda":
        torch.cuda.empty_cache()

print(f"\\nDone in {time.time()-t0:.0f}s · Output: {OUTPUT_DIR}")
if AUTO_DOWNLOAD and not PREVIEW_ONLY:
    for r in results:
        files.download(r)


---

### Tips
- **general-x4v3** + Denoise ≈ 0.4–0.6 is the sweet spot for most real-world photos/videos.
- For anime / cartoons use `anime-6B` or `anime-video-v3`.
- For 2× on clean photos, `x2plus` (or `x4plus` with Auto X2) is much faster.
- **Proteus V3** (`proteus-v3`):
  - Prefer the **1× TRT engine** you provided (`prob-v3-fgnet-fp32-576x672-1x.trt`).
  - Falls back to 2× ONNX if present.
  - Native 576×672 tiles + 32 px overlap blending → seamless results.
  - `TARGET_SCALE` is applied *after* the network (you can still do 1.5× / 2× / 4×).
- **Important**: TensorRT engines are GPU-architecture specific.  
  If the engine was built on a different card it may refuse to load on Colab T4 (sm_75).  
  In that case upload the matching ONNX or rebuild the engine on the T4.
- Dehalo 0.2–0.5 · Recover Details 0.3–0.6 · Sharpen 0.2–0.5 usually look best.

### Expected file locations
```
/content/models/prob-v3-fgnet-fp32-576x672-1x.trt   ← preferred (the file you uploaded)
/content/models/prob-v3-fgnet-fp32-576x672-2x.onnx  ← optional fallback
```

Enjoy ✨
